In [0]:
# Import necessary libraries
import sys
import os
import joblib
import time
from databricks.sdk import WorkspaceClient

# --- [CONFIGURATION] ---

# 1. The ID of the Job manually created in the UI
MANUAL_JOB_ID = 22535061017216

# 2. Path to the DiseaseTree data file (for generating the node list)
JOBLIB_PATH = "/Workspace/9900-f18a-cake/mt-method2/data/freeze0525/diseaseTree_mapped.joblib"

# 3. Initialize the Databricks SDK Client
#    It automatically uses your notebook's context for authentication.
w = WorkspaceClient()

print(f" Configuration loaded. Target Job ID: {MANUAL_JOB_ID}")

In [0]:
%pip install -r /Workspace/9900-f18a-cake/working_branch/requirements.txt

In [0]:
sys.path.append("/Workspace/9900-f18a-cake/mt-method2/src")
print("Loading DiseaseTree to get the list of nodes...")

try:
    tree_object = joblib.load(JOBLIB_PATH)
    print(" DiseaseTree loaded successfully.")
except Exception as e:
    print(f" Failed to load DiseaseTree: {e}")
    tree_object = None

def get_nodes_to_train(tree):
    """
    Traverses the tree object to extract a list of all node names.
    """
    all_nodes = []
    if tree is None: return []
    
    def traverse(node):
        # Only include non-root nodes that have samples
        if node.name != 'ZERO2' and hasattr(node, 'samples') and len(node.samples) > 0:
             all_nodes.append(node.name)
        # Recurse into children
        if hasattr(node, 'children'):
            for child in node.children:
                traverse(child)
    
    traverse(tree)
    return list(set(all_nodes))

# Get the full list of nodes from the file
nodes_to_train = get_nodes_to_train(tree_object)

# Option A: Test with a single node (Recommended first)
nodes_to_run = [  
    "Leukaemia",                   
    "Acute myeloid leukaemia",    
    "Myeloproliferative neoplasm"                     
    ] 

# Option B: Run ALL nodes (Uncomment for production)
# nodes_to_run = nodes_to_train

print(f"--- Ready to trigger {len(nodes_to_run)} tasks ---")

In [0]:
print(f"--- Triggering Job ID {MANUAL_JOB_ID} for {len(nodes_to_run)} nodes ---")

triggered_runs = []

for node in nodes_to_run:
    print(f" Triggering task for node: {node} ...")
    
    try:
        # Use 'run_now' to start a new run
        run = w.jobs.run_now(
            job_id=MANUAL_JOB_ID,
            notebook_params={
                "NODE_ID": node
            }
        )
        
        print(f"  Successfully Triggered! Run ID: {run.run_id}")
        
        triggered_runs.append(run.run_id)
        
        # Small sleep to avoid hitting API rate limits
        time.sleep(0.5)
        
    except Exception as e:
        print(f"    Failed to trigger: {e}")

print("-" * 30)
print(f"--- [COMPLETE] Triggered {len(triggered_runs)} runs. ---")
print(f" Please go to 'Workflows' -> 'Job runs' -> Click Job {MANUAL_JOB_ID} to view progress.")

In [0]:

from mch.config.settings import main_tree

def print_tree(node, level=0):
    print("  " * level + f"- {node.name}")
    for child in node.children:
        print_tree(child, level + 1)

leukaemia_node = main_tree.find_node_by_name("Leukaemia")
if leukaemia_node:
    print(" Leukaemia ：")
    print_tree(leukaemia_node)
else:
    print(" No Leukaemia Node found ")